# Inventory Variance Investigation

**Synthetic data only.** This notebook uses a deterministic miniature WMS and contains no real employer records or proprietary system details.

## tl;dr

In [ ]:
import os
from pathlib import Path

from IPython.display import Image, Markdown, display

from operational_variance_toolkit.reporting.frozen import (
    load_frozen_statistics,
    load_reconciliation_summary,
    reporting_summary,
)

statistics = load_frozen_statistics(Path(os.environ["OVT_STATISTICS_DIR"]))
reconciliation = load_reconciliation_summary(Path(os.environ["OVT_RECONSTRUCTION_DIR"]))
reporting_dir = Path(os.environ["OVT_REPORTING_DIR"])
summary = reporting_summary(statistics, reconciliation)

display(
    Markdown(
        f"**Observed result.** {summary['target_selector']} had a crude "
        f"target-minus-peer short-line difference of "
        f"{summary['crude_risk_difference_pp']:.2f} percentage points. "
        f"After the frozen measured-exposure adjustment, the difference was "
        f"{summary['adjusted_risk_difference_pp']:.2f} points "
        f"(95% interval {summary['adjusted_ci_lower_pp']:.2f} to "
        f"{summary['adjusted_ci_upper_pp']:.2f}). The evidence does not support a "
        f"generalized individual-accuracy conclusion."
    )
)

## Context & Methods

The operational question is whether a raw selector-level short signal survives comparison at a common measured work mix. The unit is an eligible pick line over the frozen simulation period. The package validates the Phase 5 manifest and checksums, then exposes accepted summary values. This notebook does not fit a model or redefine a metric.

### Key Assumptions

Assignment is observational. Recorded WMS confirmation time is not physical availability time. QA-adjustment proximity identifies candidates, not causal links.

## Data

In [ ]:
display(
    Markdown(
        f"Run `{summary['run_id']}` uses schema `{summary['schema_version']}` and "
        f"contains **{summary['eligible_picks']:,} eligible picks**, "
        f"**{summary['requested_cases']:,} requested cases**, "
        f"**{summary['short_lines']} short lines**, and "
        f"**{summary['short_cases']} short cases**. Source identity, analytical "
        f"grain, "
        f"and frozen-output checks passed. Ordinary analysis loaded ground truth: "
        f"**{summary['ordinary_analysis_ground_truth_loaded']}**."
    )
)

## Results

### 1. Three-Way Reconciliation

In [ ]:
display(
    Markdown(
        f"All **{summary['reconciliation_rows']} locations** reconcile with "
        f"**{summary['reconciliation_difference_rows']} difference rows**. "
        f"Transaction replay, live inventory, and closing snapshot each close at "
        f"**{summary['live_closing_cases']:,} cases**. Eventual recorded "
        f"agreement does "
        f"not establish point-in-time physical availability."
    )
)

### 2. Crude and Adjusted Selector Contrast

In [ ]:
display(Image(filename=str(reporting_dir / "figures" / "selector_crude_adjusted.png")))

### 3. Recorded Operational Context

In [ ]:
display(Image(filename=str(reporting_dir / "figures" / "operational_context_short_rates.png")))
display(
    Markdown(
        f"Within 120 minutes of recorded confirmation, "
        f"{summary['replenishment_near_shorts']} of "
        f"{summary['replenishment_near_picks']} picks were short; outside that "
        f"window, "
        f"{summary['replenishment_away_shorts']} of "
        f"{summary['replenishment_away_picks']} were short. This is WMS context, not "
        f"an observation of physical delay."
    )
)

### 4. QA and Adjustment Candidates

In [ ]:
display(Image(filename=str(reporting_dir / "figures" / "qa_adjustment_candidates.png")))

### 5. Sensitivity Checks

In [ ]:
display(Image(filename=str(reporting_dir / "figures" / "selector_sensitivity.png")))

### 6. Hypothesis Evidence Matrix

In [ ]:
hypothesis_lines = ["| Hypothesis | Disposition | Evidence status |", "|---|---|---|"]
for row in statistics.tables["hypothesis_evidence.csv"]:
    hypothesis_lines.append(
        f"| {row['hypothesis_id']} | {row['disposition']} | {row['evidence_status']} |"
    )
display(Markdown("\n".join(hypothesis_lines)))

## Takeaways

In [ ]:
display(
    Markdown(
        "1. **Investigate the process point first.** Observe replenishment "
        "handoff in localized high-velocity chilled work and track eligible-pick "
        "denominators.\n"
        "2. **Trace QA candidates without presuming cause.** Complete "
        "disposition review "
        "before changing controls.\n"
        "3. **Do not use raw selector totals for discipline or scoring.** The "
        "primary adjusted contrast is small and imprecise, and the ITEM-0005 "
        "sensitivity exposes "
        "limited overlap."
    )
)